# PEMS-SF 1D-CNN 训练笔记 (V2.1.3 - 7类全周 + 梯度特征)
目标： 利用 1D-CNN 的局部特征提取能力，结合原始流量与梯度信息，解决周二/周三混淆问题。

数据： PEMS-SF，双通道输入 (Raw Flow + Gradient)。

实验设计： 7分类 (0-6)，无 Kalman 平滑。

方案三：特征融合 (Global Statistics)  
原理： CNN 擅长看“形状”，但经过多次池化后，可能会丢失“绝对数值”的大小。  
周五可能比周四整体流量大 5%。  
周六虽然形状像周日，但周六的均值可能比周日高。  
我们可以手动计算几个统计特征（均值、最大值、标准差），直接拼接到全连接层前。这叫“给模型开卷考试”。

In [ ]:
# ==========================================
# Cell 1: 基础依赖与配置
# ==========================================
import os, pathlib, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 配置
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = pathlib.Path('.')
# 输出目录 V2.1.0
CM_DIR = ROOT / 'confusion_matrices213'
CM_DIR.mkdir(exist_ok=True)
MODEL_DIR = ROOT / 'models213' 
MODEL_DIR.mkdir(exist_ok=True)

# 随机种子 - 保证可复现性
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print(f'Device: {DEVICE}, Output Dir: {CM_DIR}')

In [ ]:
# 训练与解释的可复现性与确定性设置
torch.manual_seed(0); np.random.seed(0); random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ==========================================
# Cell 2: 数据读取与预处理 (修改版 - 含统计特征)
# ==========================================

# 路径配置
train_path = ROOT / 'data' / 'PEMS_train'
test_path  = ROOT / 'data' / 'PEMS_test'
labels_train_path = ROOT / 'data' / 'PEMS_trainlabels'
labels_test_path  = ROOT / 'data' / 'PEMS_testlabels'
stations_list_path = ROOT / 'data' / 'stations_list'
groups_index_path = ROOT / 'data' / 'processed' / 'pems_sf_groups_index.json' # 假设你有这个分组文件

# 基础解析函数
def parse_day_matrix(line: str):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if x.replace('.', '', 1).isdigit()]
        if nums: data.append(nums)
    if not data: return None
    arr = np.full((len(data), max(len(rr) for rr in data)), np.nan, dtype=float)
    for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

def iter_day_matrices(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            yield mat if mat is not None else None

def load_labels(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    return np.array([int(x) for x in txt.split() if x.isdigit()], dtype=int)

# 加载元数据
labels_train = load_labels(labels_train_path)
labels_test  = load_labels(labels_test_path)
station_ids = [int(x) for x in stations_list_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]').split() if x.isdigit()]

# 加载分组掩码
group_station_masks = {}
if groups_index_path.exists():
    processed = json.loads(groups_index_path.read_text(encoding='utf-8'))
    groups_index = processed.get('groups_index', {})
    sid_to_pos = {sid: i for i, sid in enumerate(station_ids)}
    for g in ['g1','g2','g3','g4','g5']:
        mask = np.zeros(len(station_ids), dtype=bool)
        for sid in groups_index.get(g, []):
            if sid in sid_to_pos: mask[sid_to_pos[sid]] = True
        group_station_masks[g] = mask
    print('分组掩码加载完毕。')
else:
    print('警告：未找到分组文件，将使用全量数据。')
    group_station_masks = {'all': np.ones(len(station_ids), dtype=bool)}

# === 核心修改：曲线清洗与特征工程 ===
def _process_curve(curve: np.ndarray):
    """
    输入: 原始单条曲线 (144,)
    输出: 
      - result: 双通道数据 (2, 144) -> [Normalized_Flow, Gradient]
      - stats:  统计特征 (3,) -> [Mean, Max, Std]
    """
    # 1. 长度对齐与插值 (保持不变)
    curve = curve[:144]
    if curve.shape[0] < 144:
        curve = np.pad(curve, (0, 144 - curve.shape[0]), mode='constant', constant_values=np.nan)
    idx = np.arange(curve.size)
    mask = ~np.isnan(curve)
    if mask.any():
        curve = np.interp(idx, idx[mask], curve[mask])
    curve = np.nan_to_num(curve, nan=0.0)

    # 2. 归一化 (Min-Max 到 0-1)
    min_val, max_val = curve.min(), curve.max()
    denom = max_val - min_val + 1e-8
    norm_curve = (curve - min_val) / denom
    
    # === 新增：计算统计特征 (基于归一化后的曲线) ===
    # 你也可以选择基于原始 curve 计算，但归一化后的数值(0-1)对神经网络更友好
    mean_val = np.mean(norm_curve)
    std_val  = np.std(norm_curve)
    # max_val 已经是1.0了(如果是MinMax归一化)，所以这里可以用原始的最大值并做一个缩放，
    # 或者计算其他特征，比如“非零值的比例”。
    # 为了简单有效，我们这里用：均值、标准差、以及原始流量的对数最大值(保留绝对量级信息)
    log_max_val = np.log1p(max_val) / 10.0 # 简单缩放到 0-1 左右
    
    stats = np.array([mean_val, std_val, log_max_val], dtype=np.float32)

    # 3. 计算梯度 (保持不变)
    grad_curve = np.gradient(norm_curve)
    
    # 4. 堆叠通道
    result = np.stack([norm_curve, grad_curve], axis=0)
    
    return result.astype(np.float32), stats # <--- 修改这里返回 stats

# Dataset 定义
class PEMS_CNNDataset(Dataset):
    def __init__(self, split_path: pathlib.Path, labels: np.ndarray, use_mask: np.ndarray):
        self.samples = []
        
        for day_i, mat in enumerate(iter_day_matrices(split_path)):
            if day_i >= len(labels): break
            if mat is None or mat.shape[1] < 144: continue
            
            original_label = int(labels[day_i])
            y = original_label - 1 
            if y < 0 or y > 6: continue 
            
            if use_mask.shape[0] == mat.shape[0]:
                mat = mat[use_mask, :]
            
            for sidx in range(mat.shape[0]):
                # 获取双通道数据 和 统计特征
                two_channel_seq, stats = _process_curve(mat[sidx, :144])
                
                if not np.isfinite(two_channel_seq).all(): continue
                self.samples.append((two_channel_seq, stats, y)) # <--- 存入 stats
        
        self.n = len(self.samples)
    
    def __len__(self): return self.n
    def __getitem__(self, idx):
        seq, stats, y = self.samples[idx]
        # 返回三个项：序列，统计特征，标签
        return torch.from_numpy(seq), torch.from_numpy(stats), torch.tensor(y, dtype=torch.long)

In [ ]:
# ==========================================
# Cell 3: 构建DataLoader
# ==========================================
group_loaders = {}
for g, mask in group_station_masks.items():
    print(f'准备分组 {g} 数据...')
    # Train
    ds_train_full = PEMS_CNNDataset(train_path, labels_train, mask)
    # 80/20 拆分训练验证
    n_train = int(len(ds_train_full) * 0.8)
    n_val = len(ds_train_full) - n_train
    train_subset, val_subset = random_split(ds_train_full, [n_train, n_val], 
                                            generator=torch.Generator().manual_seed(42))
    
    # Test
    ds_test = PEMS_CNNDataset(test_path, labels_test, mask)
    
    group_loaders[g] = {
        'train': DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0),
        'val':   DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=0),
        'test':  DataLoader(ds_test, batch_size=128, shuffle=False, num_workers=0)
    }
    print(f'  - Train: {len(train_subset)}, Val: {len(val_subset)}, Test: {len(ds_test)}')

## 模型定义：1D-CNN Fusion Hybrid
交通流解释：1D-CNN 擅长通过卷积核捕捉局部平移不变的波形特征（如拥堵的起始坡度），并相比 LSTM 训练效率更高。结合 5D 静态特征增强对整体结构的判别力；卡尔曼滤波作为前置/后置平滑去噪。

In [ ]:
# ==========================================
# Cell 4: 1D-CNN 模型定义 (修改版 - 接收统计特征)
# ==========================================
class Simple1DCNN(nn.Module):
    def __init__(self, in_channels=2, num_classes=7):
        super(Simple1DCNN, self).__init__()
        
        # Block 1-3 (保持不变)
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU()
        )
        
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Classification Head (修改输入维度)
        # CNN输出是 128 维
        # 统计特征是 3 维
        # 所以 Linear 的输入是 128 + 3 = 131
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(131, 64), # <--- 修改这里：128 -> 131
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x, stats): # <--- 增加 stats 参数
        # x: (Batch, 2, 144)
        # stats: (Batch, 3)
        
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        
        x = self.gap(x)       # (Batch, 128, 1)
        x = x.view(x.size(0), -1) # (Batch, 128)
        
        # === 核心修改：拼接特征 ===
        # combined shape: (Batch, 131)
        combined = torch.cat([x, stats], dim=1) 
        
        logits = self.fc(combined)
        return logits

## 多组训练循环与评估
- 对每个分组分别训练模型，使用验证集评估。
- 不再进行参数加权平均，而是基于分组路由的 MoE 评估逻辑：测试时，根据数据属于哪个组，调用该组对应的专家模型进行预测。

In [ ]:
# ==========================================
# Cell 5: 训练循环 (修改版 - 传递统计特征)
# ==========================================
def train_model(loaders, group_name, epochs=30):
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_acc = 0.0
    best_state = None
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        # 修改这里：解包三个变量
        for x, stats, y in loaders['train']:
            x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
            
            optimizer.zero_grad()
            out = model(x, stats) # <--- 传入 stats
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            # 修改这里：解包三个变量
            for x, stats, y in loaders['val']:
                x, stats, y = x.to(DEVICE), stats.to(DEVICE), y.to(DEVICE)
                out = model(x, stats) # <--- 传入 stats
                preds = torch.argmax(out, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        val_acc = correct / total if total > 0 else 0
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = model.state_dict()
            
        if epoch % 5 == 0:
            print(f"[{group_name}] Epoch {epoch}/{epochs} | Loss: {total_loss/len(loaders['train']):.4f} | Val Acc: {val_acc:.2%}")
            
    return best_state, best_acc

## 评估与阈值
混淆矩阵

In [ ]:
# ==========================================
# Cell 6: 可视化 - 组合混淆矩阵 (修改版 - 适配统计特征)
# ==========================================

def get_predictions(model_state, loader):
    """
    辅助函数：获取真实标签和预测标签
    修改点：适配方案三，解包 stats 并传入模型
    """
    # 初始化模型 (确保 Cell 4 的类定义已运行)
    model = Simple1DCNN(in_channels=2, num_classes=7).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()
    
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        # === 修改这里：解包三个变量 (x, stats, y) ===
        for x, stats, y in loader:
            x = x.to(DEVICE)
            stats = stats.to(DEVICE) # 别忘了把 stats 也送到 GPU
            
            # === 修改这里：传入 stats ===
            logits = model(x, stats)
            preds = torch.argmax(logits, dim=1)
            
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            
    return np.array(y_true), np.array(y_pred)

def plot_combined_matrix(mode='test'):
    """
    mode: 'test' 或 'val'
    自动遍历 G1-G5，并计算 Global，画在一张图上 (2x3)
    """
    print(f"正在生成 {mode} 集混淆矩阵组合图 (含统计特征)...")
    
    # 定义子图位置：G1-G5 占前5个，Global 占第6个
    layout = {
        'g1': (0, 0), 'g2': (0, 1), 'g3': (0, 2),
        'g4': (1, 0), 'g5': (1, 1), 'global': (1, 2)
    }
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 14))
    labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    global_true = []
    global_pred = []
    
    # 遍历 G1-G5
    for g in ['g1', 'g2', 'g3', 'g4', 'g5']:
        if g not in group_models: 
            continue
        
        # 获取预测
        loader = group_loaders[g][mode]
        # 调用修改后的 get_predictions
        y_t, y_p = get_predictions(group_models[g], loader)
        
        # 收集 Global 数据
        global_true.extend(y_t)
        global_pred.extend(y_p)
        
        # 绘图
        cm = confusion_matrix(y_t, y_p)
        # 按行归一化
        cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        row, col = layout[g]
        ax = axes[row, col]
        sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
        ax.set_title(f"{g.upper()} ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')

    # 绘制 Global
    if len(global_true) > 0:
        cm_glob = confusion_matrix(global_true, global_pred)
        cm_glob_norm = cm_glob.astype('float') / cm_glob.sum(axis=1)[:, np.newaxis]
        
        row, col = layout['global']
        ax = axes[row, col]
        sns.heatmap(cm_glob_norm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, cbar=True, ax=ax)
        ax.set_title(f"Global All Groups ({mode.capitalize()})", fontsize=14, fontweight='bold')
        ax.set_ylabel('True')
        ax.set_xlabel('Predicted')
    
    # 保存
    plt.suptitle(f'{mode.capitalize()} Set Confusion Matrices (1D-CNN + Stats, 7-Class)', 
                 fontsize=18, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_path = CM_DIR / f'combined_{mode}_matrices_v213_stats.png'
    plt.savefig(save_path, dpi=300)
    print(f"图表已保存: {save_path}")
    plt.close()

# 执行绘图
plot_combined_matrix('val')
plot_combined_matrix('test')